In [1]:
import requests
import json
import sqlite3

#API endpoit
url = "https://civicdb.org/api/graphql"
headers = {
    "Content-Type": "application/json",
}


## Fetch Variants

In [2]:

query = """
query browseVariants($after: String) {
  variants(first: 300, after: $after) {
    nodes {
      id
      name
    }
    pageInfo {
      endCursor
      hasNextPage
    }
    totalCount
  }
}
"""

all_variants = []
variables = {"after": None}

while True:
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    response_json = response.json()
    
    if 'data' in response_json:
        variants = response_json["data"]["variants"]["nodes"]
        all_variants.extend(variants)
        
        page_info = response_json["data"]["variants"]["pageInfo"]
        if not page_info["hasNextPage"]:
            break
        variables["after"] = page_info["endCursor"]
    else:
        print("Error in response:", response_json.get('errors'))
        break

print(f"Total profiles fetched: {len(all_variants)}")

Total profiles fetched: 4681


## Fetch Genes


In [3]:
query = """
query browseGenes($after: String) {
    genes(first: 300, after: $after) {
        nodes {
            id
            name
            description
            variants {
                nodes {
                    id
                    name
                    molecularProfiles {
                        nodes {
                            name
                            description
                            assertions {
                                nodes {
                                    name
                                    description
                                    disease {  # <-- Include this part
                                        id
                                        name
                                    }
                                }
                            }
                        }
                    }
                }
            }
        }
        pageInfo {
            endCursor
            hasNextPage
        }
        totalCount
    }
}
"""


all_genes = []
variables = {"after": None}

while True:
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    response_json = response.json()
    
    if 'data' in response_json:
        genes = response_json["data"]["genes"]["nodes"]
        all_genes.extend(genes)
        
        page_info = response_json["data"]["genes"]["pageInfo"]
        if not page_info["hasNextPage"]:
            break
        variables["after"] = page_info["endCursor"]
    else:
        print("Error in response:", response_json.get('errors'))
        break

print(f"Total genes fetched: {len(all_genes)}")

Total genes fetched: 710


In [4]:
#checking the disease content (mostly empty)
i = 0
for gene in all_genes:
    for variant in gene["variants"]["nodes"]:
        for profile in variant["molecularProfiles"]["nodes"]:
            if profile["assertions"]["nodes"] and profile["assertions"]["nodes"][0]["disease"]:
                    i = i+1
                    print(f"Gene: {gene['name']}, Profile: {profile['name']}, Disease: {profile['assertions']['nodes'][0]['disease']}")
print(i)

Gene: ACVR1, Profile: ACVR1 G328V, Disease: {'id': 2950, 'name': 'Diffuse Midline Glioma, H3 K27M-mutant'}
Gene: BCOR, Profile: BCOR ITD , Disease: {'id': 3408, 'name': 'Central Nervous System Tumor With BCOR Internal Tandem Duplication'}
Gene: BRAF, Profile: BRAF V600E, Disease: {'id': 7, 'name': 'Melanoma'}
Gene: BRAF, Profile: BRAF V600K, Disease: {'id': 7, 'name': 'Melanoma'}
Gene: CD44, Profile: CD44 CD44v6, Disease: {'id': 216, 'name': 'Cancer'}
Gene: CEBPA, Profile: CEBPA Mutation, Disease: {'id': 3502, 'name': 'Acute Myeloid Leukemia With CEBPA Mutation'}
Gene: CTNNB1, Profile: CTNNB1 Exon 3 Mutation, Disease: {'id': 361, 'name': 'Medulloblastoma'}
Gene: DDX41, Profile: DDX41 G530D, Disease: {'id': 224, 'name': 'Myeloid Neoplasm'}
Gene: DICER1, Profile: DICER1 RNase IIIb Mutation AND DICER1 Loss-of-function, Disease: {'id': 3431, 'name': 'Gynandroblastoma'}
Gene: DICER1, Profile: DICER1 RNase IIIb Mutation AND DICER1 Loss-of-function, Disease: {'id': 3431, 'name': 'Gynandroblas

##### store genes in SQL db

In [5]:
#store genes

# Connect to the SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('../database.db')

with open('../genomics.sql') as f:
        conn.executescript(f.read())

cursor = conn.cursor()
# Insert the genes into the table
for node in all_genes:
    disease_ids = []
    variant_ids = []
    for variant in node["variants"]["nodes"]:
        variant_ids.append(variant["id"])
        for profile in variant["molecularProfiles"]["nodes"]:
            for assertion in profile["assertions"]["nodes"]:
                if assertion["disease"] and assertion["disease"]["id"]:
                    disease_ids.append(assertion["disease"]["id"])
    disease_ids = json.dumps(list(set(disease_ids)))  # Remove duplicates and convert to JSON format
    variant_ids = json.dumps(list(set(variant_ids)))  # Remove duplicates and convert to JSON format
    cursor.execute('''
    INSERT OR REPLACE INTO genes (id, name, description, variants, diseases, db_source)
    VALUES (?, ?, ?, ?, ?, ?)
    ''', (node['id'], node['name'], node['description'], variant_ids, disease_ids, "civic"))

# Commit the transaction and close the connection
conn.commit()
conn.close()

# Fetch diseases

In [6]:
query = """
query browseDiseases($after: String) {
  diseases(first: 300, after: $after) {
    nodes {
        id
        name
    }  
    pageInfo {
      endCursor
      hasNextPage
    }
    totalCount
  }
}
"""

all_diseases = []
variables = {"after": None}

while True:
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    response_json = response.json()
    
    if 'data' in response_json:
        diseases = response_json["data"]["diseases"]["nodes"]
        all_diseases.extend(diseases)
        
        page_info = response_json["data"]["diseases"]["pageInfo"]
        if not page_info["hasNextPage"]:
            break
        variables["after"] = page_info["endCursor"]
    else:
        print("Error in response:", response_json.get('errors'))
        break

print(f"Total profiles fetched: {len(all_diseases)}")

Total profiles fetched: 831


#### store diseases in SQL db

In [7]:
# Connect to the SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('../database.db')

with open('../genomics.sql') as f:
        conn.executescript(f.read())

cursor = conn.cursor()
# Insert the filtered molecular profiles into the table
for disease in all_diseases:
    cursor.execute('''
    INSERT OR REPLACE INTO diseases (id, name, db_source)
    VALUES (?, ?, ?)
    ''', (disease['id'], disease['name'], "civic"))

# Commit the transaction and close the connection
conn.commit()
conn.close()

# Fetch Molecular Profiles

In [8]:
query = """
query browseMolecularProfiles($after: String) {
  molecularProfiles(first: 300, after: $after) {
    edges {
      node {
        id
        name
        description
        molecularProfileScore
        variants {
          id
          name
          feature {
            id
            name
          }
        }
        assertions {
          nodes{
            id
            name
            description
            disease{
              id
              name
            } 
          }
        } 
      }
    }
    pageInfo {
      endCursor
      hasNextPage
    }
    totalCount
  }
}
"""

all_molecular_profiles = []
variables = {"after": None}

while True:
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    response_json = response.json()
    
    if 'data' in response_json:
        molecular_profiles = response_json["data"]["molecularProfiles"]["edges"]
        all_molecular_profiles.extend(molecular_profiles)
        
        page_info = response_json["data"]["molecularProfiles"]["pageInfo"]
        if not page_info["hasNextPage"]:
            break
        variables["after"] = page_info["endCursor"]
    else:
        print("Error in response:", response_json.get('errors'))
        break

print(f"Total profiles fetched: {len(all_molecular_profiles)}")

Total profiles fetched: 5068


##### filter MP

In [9]:
#filter out MP with score 0
molecular_profiles_filtered = [edge for edge in all_molecular_profiles if edge["node"]["molecularProfileScore"] != 0]

print(f"Total filtered profiles: {len(molecular_profiles_filtered)}")

Total filtered profiles: 1668


##### Store MP in sql table

In [10]:

# Connect to the SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('../database.db')

with open('../genomics.sql') as f:
        conn.executescript(f.read())

cursor = conn.cursor()
# Insert the filtered molecular profiles into the table
for profile in molecular_profiles_filtered:
    node = profile['node']
    disease_name = node['assertions']['nodes'][0]['disease']['name'] if node['assertions']['nodes'] else None
    variants_name = node["variants"][0]["name"] if node['variants'] else None
    cursor.execute('''
    INSERT OR REPLACE INTO molecular_profiles (id, name, description, variants, disease, molecularProfileScore, db_source)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (node['id'], node['name'], node['description'], variants_name, disease_name,  node['molecularProfileScore'], "civic"))

# Commit the transaction and close the connection
conn.commit()
conn.close()

## Test Database

In [11]:
#test db
# Connect to the SQLite database
conn = sqlite3.connect('../database.db')
cursor = conn.cursor()

# Execute a query to retrieve all data from the molecular_profiles table
cursor.execute("SELECT * FROM genes")

# Fetch all rows from the executed query
rows = cursor.fetchall()

# Display the data
for gene in rows:
    print(gene)

# Close the connection
conn.close()

('4244', 'ABCB1', '', '[2915, 451, 262, 263, 404]', None, '[]', 'civic')
('16656', 'ABCC10', '', '[408]', None, '[]', 'civic')
('16535', 'ABCC11', '', '[3841]', None, '[]', 'civic')
('6906', 'ABCC3', '', '[407]', None, '[]', 'civic')
('7451', 'ABCG2', '', '[3852, 260]', None, '[]', 'civic')
('4', 'ABL1', 'ABL1 is most relevant to cancer in its role in the BCR-ABL fusion protein that has become a signature of chronic myeloid leukemia (CML). Cells harboring this fusion have shown sensitivity to imatinib, greatly improving the prognostic outlook of the disease. However, additional mutations in ABL1 have been shown to confer resistance to imatinib. In these resistance cases, second-generation tyrosine kinase inhibitors such as dasatinib and nilotinib have exhibited some efficacy and are currently undergoing clinical trials for treating acquired resistance in CML.', '[1536, 4609, 4610, 3, 1028, 1538, 4611, 1029, 1537, 1527, 2, 4632, 4634, 4635, 4636, 4637, 4638, 1023, 4639, 4645, 1535, 1595

In [104]:
#check disease content
# Connect to the SQLite database
conn = sqlite3.connect('../database.db')
cursor = conn.cursor()

# Execute a query to retrieve all data from the molecular_profiles table
cursor.execute("SELECT * FROM genes")

# Fetch all rows from the executed query
rows = cursor.fetchall()

# Display the data
i = 0
for gene in rows:
    if len(gene[5])>2:
        print(gene[5])
        i = i+1

# Close the connection
conn.close()
print(i)

[2950]
[3408, 3450, 3207]
[11, 7]
[216]
[3502]
[361]
[224]
[3432, 3466, 3431]
[368]
[51]
[216, 157]
[3]
[3225]
[2953]
[2150]
[3069]
[216]
[8]
[3224]
[3241, 3387]
[3387]
[8]
[3433]
[15]
[3465, 3516, 118, 3447]
[3]
26


In [88]:
gene[5]

'[]'

In [ ]:
#checking the assertions content (mostly empty)
i = 0
for gene in all_genes:
    for variant in gene["variants"]["nodes"]:
        for profile in variant["molecularProfiles"]["nodes"]:
            if profile["assertions"]["nodes"]:  # If assertions exist
                i = i+1
                print(f"Gene: {gene['name']}, Profile: {profile['name']}, Assertions: {profile['assertions']['nodes']}")
print(i)

In [21]:
#test retrieval by name
def get_all_molecular_profiles(db_path):
    # Connect to the SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Execute a query to retrieve all data from the molecular_profiles table
    cursor.execute("SELECT * FROM molecular_profiles")

    # Fetch all rows from the executed query
    rows = cursor.fetchall()

    # Close the connection
    conn.close()

    return rows

NAME = "BRAC2 Mutation"

print(rows)
    
    #if profile['node']['name'] == NAME:
    #     print(profile['node'])


[('12', 'BRAF V600E', 'BRAF V600E has been shown to be recurrent in many cancer types. It is one of the most widely studied variants in cancer. This variant is correlated with poor prognosis in certain cancer types, including colorectal cancer and papillary thyroid cancer. The targeted therapeutic dabrafenib has been shown to be effective in clinical trials with an array of BRAF mutations and cancer types. Dabrafenib has also shown to be effective when combined with the MEK inhibitor trametinib in colorectal cancer and melanoma. However, in patients with TP53, CDKN2A and KRAS mutations, dabrafenib resistance has been reported. Ipilimumab, regorafenib, vemurafenib, and a number of combination therapies have been successful in treating V600E mutations. However, cetuximab and panitumumab have been largely shown to be ineffective without supplementary treatment.', 'V600E', None, 'Melanoma', 1433.5, 'civic'), ('302', 'ERBB2 Amplification', 'Her2 (ERBB2) amplifications are seen in up to 20% 

In [22]:
def get_all_molecular_profiles_with_keys(db_path):
    # Connect to the SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Execute a query to retrieve all data from the molecular_profiles table
    cursor.execute("SELECT * FROM molecular_profiles")

    # Fetch all rows from the executed query
    rows = cursor.fetchall()

    # Get the column names from the cursor description
    column_names = [description[0] for description in cursor.description]

    # Close the connection
    conn.close()

    # Combine column names with rows
    profiles_with_keys = [dict(zip(column_names, row)) for row in rows]

    return profiles_with_keys

# Use the function to get the profiles
profiles_with_keys = get_all_molecular_profiles_with_keys('../database.db')
print(profiles_with_keys)

[{'id': '12', 'name': 'BRAF V600E', 'description': 'BRAF V600E has been shown to be recurrent in many cancer types. It is one of the most widely studied variants in cancer. This variant is correlated with poor prognosis in certain cancer types, including colorectal cancer and papillary thyroid cancer. The targeted therapeutic dabrafenib has been shown to be effective in clinical trials with an array of BRAF mutations and cancer types. Dabrafenib has also shown to be effective when combined with the MEK inhibitor trametinib in colorectal cancer and melanoma. However, in patients with TP53, CDKN2A and KRAS mutations, dabrafenib resistance has been reported. Ipilimumab, regorafenib, vemurafenib, and a number of combination therapies have been successful in treating V600E mutations. However, cetuximab and panitumumab have been largely shown to be ineffective without supplementary treatment.', 'variants': 'V600E', 'TEXT': None, 'disease': 'Melanoma', 'molecularProfileScore': 1433.5, 'db_sou

In [23]:
profiles_with_keys

[{'id': '12',
  'name': 'BRAF V600E',
  'description': 'BRAF V600E has been shown to be recurrent in many cancer types. It is one of the most widely studied variants in cancer. This variant is correlated with poor prognosis in certain cancer types, including colorectal cancer and papillary thyroid cancer. The targeted therapeutic dabrafenib has been shown to be effective in clinical trials with an array of BRAF mutations and cancer types. Dabrafenib has also shown to be effective when combined with the MEK inhibitor trametinib in colorectal cancer and melanoma. However, in patients with TP53, CDKN2A and KRAS mutations, dabrafenib resistance has been reported. Ipilimumab, regorafenib, vemurafenib, and a number of combination therapies have been successful in treating V600E mutations. However, cetuximab and panitumumab have been largely shown to be ineffective without supplementary treatment.',
  'variants': 'V600E',
  'TEXT': None,
  'disease': 'Melanoma',
  'molecularProfileScore': 143

In [51]:
for profile in profiles_with_keys:
    print(profile['disease'])

NameError: name 'profiles_with_keys' is not defined